# SaaS Financial Dataset Reconstruction (Data Treatment)

This notebook reconstructs a **finance-oriented, executive-grade** SaaS dataset from three raw tables:
- `ravenstack_accounts.csv` (CRM account metadata)
- `ravenstack_subscriptions.csv` (contract lifecycle & revenue)
- `ravenstack_churn_events.csv` (churn records)

**Goal**: produce a single **account-level** CSV (`saas_financial_snapshot.csv`) suitable for Strategy/CFO analysis (ARR, churn, segmentation), not for product analytics.


## Objectives
- Normalize raw tables to an **account-level** view
- Extract **latest ARR** per account (finance-first simplification)
- Map **segments** (SMB / Mid-Market / Enterprise) and **regions** (NA / EMEA / APAC)
- Build a minimal **ARR bridge**: `net_arr_change = new_arr − churned_arr`
- Export a clean CSV for downstream EDA & decision engine


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:,.0f}'.format)


## 1) Load raw data
We keep paths simple so this notebook can run from the project root or `/notebooks` (adjust paths if needed).

In [3]:
ACCOUNTS_PATH = 'C:/ai-decision-intelligence-main/data/ravenstack_accounts.csv'
SUBSCRIPTIONS_PATH = 'C:/ai-decision-intelligence-main/data/ravenstack_subscriptions.csv'
CHURN_PATH = 'C:/ai-decision-intelligence-main/data/ravenstack_churn_events.csv'

accounts = pd.read_csv(ACCOUNTS_PATH)
subscriptions = pd.read_csv(SUBSCRIPTIONS_PATH)
churn_events = pd.read_csv(CHURN_PATH)

len(accounts), len(subscriptions), len(churn_events)

(500, 5000, 600)

## 2) Quick overview (sanity check)
We **inspect** structure only—no analysis here.

In [4]:
display(accounts.head())
accounts.info()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   account_id       500 non-null    object
 1   account_name     500 non-null    object
 2   industry         500 non-null    object
 3   country          500 non-null    object
 4   signup_date      500 non-null    object
 5   referral_source  500 non-null    object
 6   plan_tier        500 non-null    object
 7   seats            500 non-null    int64 
 8   is_trial         500 non-null    bool  
 9   churn_flag       500 non-null    bool  
dtypes: bool(2), int64(1), object(7)
memory usage: 32.4+ KB


In [5]:
display(subscriptions.head())
subscriptions.info()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   subscription_id    5000 non-null   object
 1   account_id         5000 non-null   object
 2   start_date         5000 non-null   object
 3   end_date           486 non-null    object
 4   plan_tier          5000 non-null   object
 5   seats              5000 non-null   int64 
 6   mrr_amount         5000 non-null   int64 
 7   arr_amount         5000 non-null   int64 
 8   is_trial           5000 non-null   bool  
 9   upgrade_flag       5000 non-null   bool  
 10  downgrade_flag     5000 non-null   bool  
 11  churn_flag         5000 non-null   bool  
 12  billing_frequency  5000 non-null   object
 13  auto_renew_flag    5000 non-null   bool  
dtypes: bool(5), int64(3), object(6)
memory usage: 376.1+ KB


In [6]:
display(churn_events.head())
churn_events.info()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,55,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0,False,True,True,too expensive


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    object 
 1   account_id                600 non-null    object 
 2   churn_date                600 non-null    object 
 3   reason_code               600 non-null    object 
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    object 
dtypes: bool(3), float64(1), object(5)
memory usage: 30.0+ KB


## 3) Business assumptions (explicit)
These assumptions make the dataset **decision-ready** and are typical in Strategy/CFO work:

1. **Unit of analysis**: one row = one **customer account**
2. **Revenue metric**: **ARR** from the **latest subscription** per account (annualized revenue proxy)
3. **Segmentation by plan**:
   - `Basic` → **SMB**
   - `Pro` → **Mid-Market**
   - `Enterprise` → **Enterprise**
4. **Geographic aggregation** (exec-level): `NA`, `EMEA`, `APAC` from country codes
5. **Gross margin (estimated)** by segment (benchmark-inspired):
   - SMB: 65%
   - Mid-Market: 75%
   - Enterprise: 82%
6. **ARR bridge (minimal)**: `net_arr_change = new_arr − churned_arr` where `new_arr` is current ARR; churned accounts lose their ARR.

> Note: expansion/contraction are set to 0 here **by design** and will be introduced in Phase 2 (scenario modeling).

## 4) Account-level normalization
We select essential CRM fields and create **segment** and **region** features.

In [7]:
base_df = accounts[[
    'account_id','industry','country','plan_tier'
]].copy()

region_map = {
    'US':'NA','CA':'NA',   # North America
    'UK':'EMEA','DE':'EMEA','FR':'EMEA',  # Europe, Middle East & Africa
    'IN':'APAC','AU':'APAC'  # Asia-Pacific
}
segment_map = {'Basic':'SMB','Pro':'Mid-Market','Enterprise':'Enterprise'}

base_df['region'] = base_df['country'].map(region_map)
base_df['segment'] = base_df['plan_tier'].map(segment_map)

base_df.head()

,account_id,industry,country,plan_tier,region,segment
0,A-2e4581,EdTech,US,Basic,NA,SMB
1,A-43a9e3,FinTech,IN,Basic,APAC,SMB
2,A-0a282f,DevTools,US,Basic,NA,SMB
3,A-1f0ac7,HealthTech,UK,Basic,EMEA,SMB
4,A-ce550d,HealthTech,US,Enterprise,NA,Enterprise


## 5) Subscription normalization → latest ARR per account
We sort by `start_date` and keep the **last** record per `account_id` as a proxy for current ARR.

In [8]:
# Ensure date sort works if start_date is a string
subscriptions_sorted = subscriptions.sort_values('start_date')
latest_sub = (
    subscriptions_sorted
    .groupby('account_id')
    .last()
    .reset_index()
)

# Merge ARR onto the account table
base_df = base_df.merge(
    latest_sub[['account_id','arr_amount']],
    on='account_id',
    how='left'
)
base_df = base_df.rename(columns={'arr_amount':'annual_contract_value'})
base_df['annual_contract_value'] = base_df['annual_contract_value'].fillna(0)

base_df.head()

,account_id,industry,country,plan_tier,region,segment,annual_contract_value
0,A-2e4581,EdTech,US,Basic,NA,SMB,10032
1,A-43a9e3,FinTech,IN,Basic,APAC,SMB,10584
2,A-0a282f,DevTools,US,Basic,NA,SMB,1176
3,A-1f0ac7,HealthTech,UK,Basic,EMEA,SMB,14112
4,A-ce550d,HealthTech,US,Enterprise,NA,Enterprise,260292


## 6) Gross margin estimation (segment-based)
We apply benchmark-style gross margin per segment to enable unit economics later (e.g., contribution by segment).

In [9]:
gross_margin_map = {'SMB':0.65,'Mid-Market':0.75,'Enterprise':0.82}
base_df['gross_margin_estimated'] = base_df['segment'].map(gross_margin_map)

base_df[['segment','gross_margin_estimated']].head()

,segment,gross_margin_estimated
0,SMB,1
1,SMB,1
2,SMB,1
3,SMB,1
4,Enterprise,1


## 7) Churn normalization → churned ARR
If an account appears in `churn_events`, we treat its **current ARR** as **lost** for the bridge.

In [10]:
churned_accounts = churn_events['account_id'].unique()
base_df['churned_arr'] = np.where(
    base_df['account_id'].isin(churned_accounts),
    base_df['annual_contract_value'],
    0
)
base_df['churned_arr'].head()

0     10032
1         0
2      1176
3     14112
4    260292
Name: churned_arr, dtype: int64

## 8) Minimal ARR bridge (board-style)
We keep a **clean** bridge for Phase 1:
- `new_arr` = current annual ARR
- `expansion_arr` = 0 (to be modeled later)
- `contraction_arr` = 0 (to be modeled later)
- `net_arr_change = new_arr − churned_arr`

In [11]:
base_df['new_arr'] = np.where(base_df['annual_contract_value']>0, base_df['annual_contract_value'], 0)
base_df['expansion_arr'] = 0
base_df['contraction_arr'] = 0
base_df['net_arr_change'] = base_df['new_arr'] - base_df['churned_arr']

final_df = base_df[[
    'account_id','segment','region','industry',
    'annual_contract_value','gross_margin_estimated',
    'new_arr','expansion_arr','contraction_arr','churned_arr','net_arr_change'
]]

final_df.head()

,account_id,segment,region,industry,annual_contract_value,gross_margin_estimated,new_arr,expansion_arr,contraction_arr,churned_arr,net_arr_change
0,A-2e4581,SMB,NA,EdTech,10032,1,10032,0,0,10032,0
1,A-43a9e3,SMB,APAC,FinTech,10584,1,10584,0,0,0,10584
2,A-0a282f,SMB,NA,DevTools,1176,1,1176,0,0,1176,0
3,A-1f0ac7,SMB,EMEA,HealthTech,14112,1,14112,0,0,14112,0
4,A-ce550d,Enterprise,NA,HealthTech,260292,1,260292,0,0,260292,0


## 9) Sanity checks
Basic validations to ensure the snapshot is coherent for EDA and decision engine.

In [14]:
print('Shape:', final_df.shape)
print('Nulls per column:')
print(final_df.isna().sum())
print('ARR by segment (5-number summary):')
display(final_df.groupby('segment')['annual_contract_value'].describe())
print('Top 5 accounts by ARR:')
display(final_df.nlargest(5, 'annual_contract_value'))

Shape: (500, 11)
Nulls per column:
account_id                0
segment                   0
region                    0
industry                  0
annual_contract_value     0
gross_margin_estimated    0
new_arr                   0
expansion_arr             0
contraction_arr           0
churned_arr               0
net_arr_change            0
dtype: int64
ARR by segment (5-number summary):


,count,mean,std,min,25%,50%,75%,max
segment,,,,,,,,
Enterprise,154,"32,761","52,117",0,"3,672","11,850","38,208","389,244"
Mid-Market,178,"27,904","36,555",0,"4,389","14,802","38,208","250,740"
SMB,168,"27,443","41,674",0,"3,363","10,014","31,368","279,396"


Top 5 accounts by ARR:


,account_id,segment,region,industry,annual_contract_value,gross_margin_estimated,new_arr,expansion_arr,contraction_arr,churned_arr,net_arr_change
177,A-56962b,Enterprise,EMEA,EdTech,389244,1,389244,0,0,389244,0
403,A-d4e0d4,SMB,NA,FinTech,279396,1,279396,0,0,0,279396
4,A-ce550d,Enterprise,NA,HealthTech,260292,1,260292,0,0,260292,0
368,A-1f0636,Mid-Market,APAC,EdTech,250740,1,250740,0,0,0,250740
130,A-5c046d,SMB,EMEA,EdTech,222084,1,222084,0,0,0,222084


## 10) Export curated dataset
This will create `saas_financial_snapshot.csv` in the working directory.

In [15]:
OUTPUT_PATH = 'saas_financial_snapshot.csv'
final_df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

'saas_financial_snapshot.csv'

## 11) Limitations & next steps
- Snapshot is **reconstructed**, not transactional truth.
- Expansion/contraction set to 0 **by design** to keep Phase 1 focused.
- Phase 2 will introduce: expansion modeling, contraction, pricing sensitivity, and scenario analysis under constraints.

> You can now start **Phase 1 — Hypothesis‑Driven EDA** using `saas_financial_snapshot.csv`.
